In [1]:
import warnings
warnings.filterwarnings("ignore")
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np



Instructions for updating:
non-resource variables are not supported in the long term


순환 신경망(Recurrent Neural Network, RNN)

시계열 데이터(Time Series)나 자연어(Natural Language)와 같이 순서가 있는 데이터(Sequential Data)를 처리하기 위해 특화된 인공신경망 구조이다.  
RNN은 순차적인 데이터를 입력받아 결과값을 도출하는데 사용하는 딥러닝 모델로 자연어 처리에 상당히 많이 사용되고 이전에 입력된 값들을 고려해서 현재 입력값의 출력값을 결정하는 딥러닝 모델이다.

<img src="./RNN.png" wigth="900" align="left" />

x는 입력값, y는 출려값, 활성화 함수(tanh)를 거친 값은 상태(출력)을 의미한다. 네모 상자는 셀이라 하며 셀 안에서 현재 셀의 입력값과 과거 셀의 상태값을 사용해서 현재 셀의 상태값을 계산한다.  
현재 셀의 상태값은 현재 셀의 출력값과 동일하며 다음 셀의 이전 상태값으로 사용된다.

상태값을 결정하기 위해서는 그림과 같이 두 가지의 가중치가 존재한다. 현재 셀의 상태값은 tanh(입력값 * W<sub>xh</sub> + 이전 셀의 상태값 * W<sub>hh</sub> + 바이어스)으로 결정된다. 가중치와 바이어스는 최초 셀에는 무작위로 부여하고 학습 과정을 통해 가중치 및 바이어스는 목적에 맞에 최적화 된다.

tensorflow로 RNN 구현하기

BasicRNNCell은 tenserflow 2.15 버전까지 사용할 수 있다. tenserflow 2.16 버전 부터는 keras 3.x이 설치되기때문에 BasicRNNCell을 사용할 수 없다.  
현재 수업하는 시점에서 tenserflow 2.21 버전이 자동 설치되므로 pip uninstall tensorflow 명령을 먼저 실행해서 tenserflow 2.21 버전을 제거하고 pip uninstall tensorflow==2.15 명령을 실행해서 tenserflow 2.15 버전을 설치한다.

!pip uninstall tensorflow  
!pip install tensorflow==2.15

In [2]:
inputs = np.array([[[1, 2]]]) # RNN 입력 데이터
# print(inputs)
# print(inputs.shape)
# print(inputs.shape[0])

rnn_inputs = tf.constant(inputs, dtype=tf.float32)
sess = tf.Session()
print('입력 데이터: {}'.format(sess.run(rnn_inputs)))
print('=' * 100)

# BasicRNNCell() 객체를 생성할 때 num_units 속성값으로 RNN 셀의 개수를 지정해서 생성한다.
rnn_cell = tf.nn.rnn_cell.BasicRNNCell(num_units=4)
print('RNN 셀의 개수: {}'.format(rnn_cell.state_size))
print('=' * 100)

# dynamic_rnn() 메소드는 입력값과 RNN 셀을 받아서 실행 결과(상태값, 출력값)를 리턴한다.
# dynamic_rnn(cell, inputs, dtype)
# cell: RNN 셀 객체, inputs: RNN 셀에 입력되는 데이터, dtype: RNN 셀에 입력되는 데이터의 타입
outputs, state = tf.nn.dynamic_rnn(cell=rnn_cell, inputs=rnn_inputs, dtype=tf.float32)
print('=' * 100)
print('출력값: {}'.format(outputs))
print('상태값: {}'.format(state))
print('=' * 100)

print('가중치의 개수와 바이어스의 개수')
for v in tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES):
    print(v)

입력 데이터: [[[1. 2.]]]

RNN 셀의 개수: 4
Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
출력값: Tensor("rnn/transpose_1:0", shape=(1, 1, 4), dtype=float32)
상태값: Tensor("rnn/while/Exit_3:0", shape=(1, 4), dtype=float32)
가중치의 개수와 바이어스의 개수
<tf.Variable 'rnn/basic_rnn_cell/kernel:0' shape=(6, 4) dtype=float32_ref>
<tf.Variable 'rnn/basic_rnn_cell/bias:0' shape=(4,) dtype=float32_ref>


In [3]:
var_name = [v.name for v in tf.trainable_variables()]
print(var_name)

with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    
    _outputs, _state = sess.run([outputs, state])
    # 출력값은 상태값과 같은 말이고 상태값은 다음 RNN 셀로 전달된다.
    print('출력값: {}'.format(_outputs))
    print('상태값: {}'.format(_state))
    print('=' * 100)
    
    values = sess.run(var_name)
    # 가중치 행 개수는 '입력 데이터 피쳐 크기 + RNN 셀 개수'이므로 6개이고 열 개수는 'RNN 셀 개수'이므로 4개이다.
    print('가중치\n', var_name[0], '\n', values[0], sep='')
    # 바이어스 개수는 'RNN 셀 개수'이므로 4개이다.
    print('바이어스\n', var_name[1], '\n', values[1], sep='')

['rnn/basic_rnn_cell/kernel:0', 'rnn/basic_rnn_cell/bias:0']
출력값: [[[ 0.42205867 -0.6594622  -0.00807506  0.97789305]]]
상태값: [[ 0.42205867 -0.6594622  -0.00807506  0.97789305]]
가중치
rnn/basic_rnn_cell/kernel:0
[[ 0.18459737  0.70382     0.76681066  0.7474054 ]
 [ 0.13279843 -0.7478407  -0.38744295  0.74977136]
 [-0.00349963 -0.34340993 -0.34700322  0.49370146]
 [ 0.5444838  -0.4147709  -0.36751598  0.54917955]
 [-0.39480883 -0.4337019  -0.04494661 -0.40770325]
 [ 0.2592827   0.3023535  -0.6090524   0.34866238]]
바이어스
rnn/basic_rnn_cell/bias:0
[0. 0. 0. 0.]
